# SD1.5 Prompt-Matched In-Range Recovery Results

This notebook recreates the paper-style recovery figures for the in-range sunset-beach experiment where the recovery prompt matches the target content. It loads diffusion-backprop reconstructions sampled with the configured SD1.5 k-tilde priors, merges completed aggregate CSVs with any partial per-run artifacts, filters to the sampling ratios used for the figure set, and builds a mean metric table before plotting.

Run it top to bottom after the prompt-matched suites are available under `results/`. The next cell resolves the project root, imports `sd15_conditioning_experiment.py`, chooses the first non-empty result-tag group, reports which source tags were actually loaded, and prepares `results/analysis/prompt_matched_in_range` for the exported figures.

In [ ]:
from pathlib import Path
import importlib
import math
import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
for search_root in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    helper_dir = search_root / 'analyze_results'
    helper_path = helper_dir / 'sd15_conditioning_experiment.py'
    if helper_path.exists():
        if str(helper_dir) not in sys.path:
            sys.path.insert(0, str(helper_dir))
        break
    for child in search_root.iterdir():
        if not child.is_dir():
            continue
        helper_dir = child / 'analyze_results'
        helper_path = helper_dir / 'sd15_conditioning_experiment.py'
        if helper_path.exists():
            if str(helper_dir) not in sys.path:
                sys.path.insert(0, str(helper_dir))
            break
    else:
        continue
    break
else:
    raise FileNotFoundError('Could not find sd15_conditioning_experiment.py from the notebook cwd.')

import sd15_conditioning_experiment as exp
exp = importlib.reload(exp)

SD15_ROOT = exp.find_sd15_root(NOTEBOOK_DIR)
SPLIT_FINAL_TAGS = [
    'prompt_matched_in_range_sample_k0_unconditioned',
    'prompt_matched_in_range_sample_k1_daytime_beach',
    'prompt_matched_in_range_sample_k2_sunset_beach',
    'prompt_matched_in_range_sample_k4_cat',
]
TAG_GROUP_CANDIDATES = [
    ('prompt_matched_in_range', SPLIT_FINAL_TAGS),
    ('prompt_matched_in_range', ['prompt_matched_in_range']),
]
DC_METHODS = ['diffusion_backprop']
SAMPLING_METHODS = ['cs']
ALLOWED_SAMPLING_PERC = {0.00125, 0.0025, 0.005, 0.01, 0.025,}
EXCLUDED_SAMPLING_CONDITIONS = set()
OUTPUT_ROOT = SD15_ROOT / 'results' / 'analysis'

ROWS = pd.DataFrame()
ACTIVE_TAG = TAG_GROUP_CANDIDATES[0][0]
LOADED_TAGS = []
for output_tag, tags in TAG_GROUP_CANDIDATES:
    frames = []
    loaded_tags = []
    for tag in tags:
        try:
            candidate_rows = exp.load_regression_rows(
                SD15_ROOT,
                tag=tag,
                dc_methods=DC_METHODS,
                sampling_methods=SAMPLING_METHODS,
                include_partial=True,
            )
        except FileNotFoundError:
            candidate_rows = pd.DataFrame()
        if candidate_rows.empty:
            continue
        candidate_rows = candidate_rows.copy()
        candidate_rows['source_suite_tag'] = tag
        frames.append(candidate_rows)
        loaded_tags.append(tag)
    if frames:
        ROWS = pd.concat(frames, ignore_index=True)
        ACTIVE_TAG = output_tag
        LOADED_TAGS = loaded_tags
        break

OUTPUT_DIR = OUTPUT_ROOT / ACTIVE_TAG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not ROWS.empty and ALLOWED_SAMPLING_PERC:
    allowed = np.array(sorted(float(value) for value in ALLOWED_SAMPLING_PERC), dtype=float)
    samp_values = ROWS['samp_perc'].astype(float).to_numpy()
    keep_allowed = np.isclose(samp_values[:, None], allowed[None, :], rtol=0.0, atol=5e-8).any(axis=1)
    ROWS = ROWS[keep_allowed].copy()
if not ROWS.empty and EXCLUDED_SAMPLING_CONDITIONS:
    ROWS = ROWS[~ROWS['sampling_condition'].isin(EXCLUDED_SAMPLING_CONDITIONS)].copy()
MEAN_TABLE = exp.build_mean_metric_table(ROWS) if not ROWS.empty else pd.DataFrame()

print(f'Active tag: {ACTIVE_TAG}')
print(f'Loaded source tags: {LOADED_TAGS}')
print(f'Loaded {len(ROWS)} run rows.')
display(MEAN_TABLE)
if MEAN_TABLE.empty:
    print('No recovery rows found yet. Run the suite first, then rerun this notebook.')


## Metric Curves

This cell defines the plotting style and helper functions for the recovery-prompt sweep figures, then exports metric curves for `psnr_db`, `ssim`, and `pixel_mae` plus a combined PSNR/SSIM panel for each diffusion/sampling method that has rows. Each subplot fixes a sampling prior, the colored lines compare recovery prompts, and the x-axis is the sampling ratio `m/n` on a log scale.

Curves show the mean over repeats. The shaded region is a 95% normal-approximation confidence interval, computed as mean +/- 1.96 SEM in the plotted metric units. The black dashed reference is the zero-filled inverse FFT baseline when those metrics are present.

In [ ]:
from statistics import NormalDist

PROMPT_TEXT = {
    'unprompted': '',
    'daytime_beach': 'daytime beach',
    'sunset_beach': 'sunset beach',
    'cat': 'cat',
}
METRIC_LABELS = {
    'psnr_db': 'PSNR (dB)',
    'ssim': 'SSIM',
    'pixel_mae': 'Per-Pixel MAE',
}
SHOW_METRIC_CONFIDENCE_INTERVALS = True
METRIC_CONFIDENCE_LEVEL = 0.95
GLOBAL_SAMPLING_X_LABEL = r'Sampling Ratio $m/n$'
try:
    import sd15_cfg_ablation_analysis as cfg_ablation
    cfg_ablation = importlib.reload(cfg_ablation)
    ABLATION_RECON_COLORS = [
        cfg_ablation.LINE_COLORS[key]
        for key in ['unconditioned', 'cfg1', 'cfg7p5', 'cfg5', 'cfg3']
    ]
except Exception:
    ABLATION_RECON_COLORS = ['#4C78A8', '#54A24B', '#E45756', '#6A3D9A', '#F58518']
RECON_COLOR_BY_CONDITION = {
    'unprompted': ABLATION_RECON_COLORS[0],
    'daytime_beach': ABLATION_RECON_COLORS[1],
    'sunset_beach': ABLATION_RECON_COLORS[2],
    'cat': ABLATION_RECON_COLORS[3],
}
SWEEP_FIGSIZE_PER_COL = 4.9
SWEEP_FIGSIZE_PER_ROW = 3.7
SWEEP_SINGLE_FIGSIZE_HEIGHT = 5.25
SWEEP_SINGLE_LEGEND_Y = 0.985
SWEEP_SINGLE_TOP = 0.76
SWEEP_SINGLE_BOTTOM = 0.28
SWEEP_SINGLE_LEFT = 0.065
SWEEP_SINGLE_RIGHT = 0.995
SWEEP_SINGLE_WSPACE = 0.18
SWEEP_LEGEND_Y = 1.08
SWEEP_LINEWIDTH = 2.4
SWEEP_MARKERSIZE = 7.0
SWEEP_MARKER_EDGEWIDTH = 1.3


def title_case(text):
    return ' '.join(str(text).replace('_', ' ').split()).title()


def prompt_text_for(condition, default_label=''):
    return PROMPT_TEXT.get(str(condition), str(default_label).replace(' Recovery', '').replace(' Reconstruction', '').lower())


def recovery_math_label(condition, label=None):
    prompt = prompt_text_for(condition, label or condition)
    return rf'$c_r = \texttt{{"{prompt}"}}$'


def sampling_mu_label(condition):
    return rf'${exp._mu_symbol(str(condition), hat=True)}$'


def sampling_tick_labels(values):
    labels = []
    for value in values:
        value = float(value)
        if value < 0.01:
            labels.append(f'{value:.5f}')
        elif value < 0.1:
            labels.append(f'{value:.3f}')
        else:
            labels.append(f'{value:.2f}'.rstrip('0').rstrip('.'))
    return labels


def reconstruction_colors():
    return ABLATION_RECON_COLORS


def reconstruction_color(condition_or_idx, idx=None):
    colors = reconstruction_colors()
    if isinstance(condition_or_idx, str):
        if condition_or_idx in RECON_COLOR_BY_CONDITION:
            return RECON_COLOR_BY_CONDITION[condition_or_idx]
        if idx is not None:
            return colors[int(idx) % len(colors)]
    return colors[int(condition_or_idx) % len(colors)]


def legend_zero_filled_last(handles, labels):
    paired = list(zip(handles, labels))
    ordered = [(handle, label) for handle, label in paired if label != 'Zero-Filled']
    ordered.extend((handle, label) for handle, label in paired if label == 'Zero-Filled')
    if not ordered:
        return [], []
    return [handle for handle, _ in ordered], [label for _, label in ordered]


def metric_summary(frame):
    return exp.build_metric_summary_table(frame, confidence_level=METRIC_CONFIDENCE_LEVEL)



ZERO_FILLED_METRIC_COLUMNS = {
    'psnr_db': 'zero_filled_psnr_db',
    'ssim': 'zero_filled_ssim',
    'pixel_mae': 'zero_filled_pixel_mae',
}


def zero_filled_metric_summary(frame, metric):
    ci_column = f'{metric}_ci_halfwidth'
    empty = pd.DataFrame(columns=['sampling_condition', 'sampling_rank', 'samp_perc', metric, ci_column])
    column = ZERO_FILLED_METRIC_COLUMNS.get(metric)
    if column is None or column not in frame.columns:
        return empty

    needed_columns = [
        'dc_method',
        'sampling_method',
        'sampling_condition',
        'sampling_rank',
        'samp_perc',
        'item_id',
        'repeat_id',
        column,
    ]
    available_columns = [col for col in needed_columns if col in frame.columns]
    baseline = frame[available_columns].copy()
    baseline[column] = pd.to_numeric(baseline[column], errors='coerce')
    baseline['samp_perc'] = pd.to_numeric(baseline['samp_perc'], errors='coerce')
    baseline = baseline.dropna(subset=['sampling_condition', 'samp_perc', column])
    if baseline.empty:
        return empty
    dedupe_columns = [
        col for col in ['dc_method', 'sampling_method', 'sampling_condition', 'samp_perc', 'item_id', 'repeat_id']
        if col in baseline.columns
    ]
    if dedupe_columns:
        baseline = baseline.drop_duplicates(subset=dedupe_columns, keep='last')

    group_columns = [col for col in ['sampling_condition', 'sampling_rank', 'samp_perc'] if col in baseline.columns]
    z_value = 0.0
    if METRIC_CONFIDENCE_LEVEL > 0.0:
        z_value = float(NormalDist().inv_cdf(0.5 + 0.5 * min(float(METRIC_CONFIDENCE_LEVEL), 0.999999)))
    summary = (
        baseline.groupby(group_columns, dropna=False, sort=False)[column]
        .agg(mean='mean', std='std', count='count')
        .reset_index()
    )
    summary['std'] = summary['std'].fillna(0.0)
    summary['count'] = summary['count'].fillna(0).astype(int)
    summary['sem'] = 0.0
    valid = summary['count'] > 0
    summary.loc[valid, 'sem'] = summary.loc[valid, 'std'] / np.sqrt(summary.loc[valid, 'count'].astype(float))
    summary[ci_column] = summary['sem'] * z_value
    return summary.rename(columns={'mean': metric})[group_columns + [metric, ci_column]]

def plot_metric_curves(frame, metric, output_path=None, show=True):
    if frame.empty:
        print(f'No rows available for {metric}.')
        return None

    summary = metric_summary(frame)
    sampling_cases = (
        summary[['sampling_condition', 'sampling_rank']]
        .drop_duplicates()
        .sort_values('sampling_rank', kind='stable')
    )
    recon_cases = (
        summary[['reconstruction_condition', 'reconstruction_label', 'recon_rank']]
        .drop_duplicates()
        .sort_values('recon_rank', kind='stable')
    )
    band_column = f'{metric}_ci_halfwidth'
    zero_summary = zero_filled_metric_summary(frame, metric)

    with plt.rc_context(exp.SD15_PRESENTATION_RC):
        fig, axes = plt.subplots(
            1,
            len(sampling_cases),
            figsize=(SWEEP_FIGSIZE_PER_COL * len(sampling_cases), SWEEP_SINGLE_FIGSIZE_HEIGHT),
            sharey=True,
            constrained_layout=False,
        )
        axes = np.atleast_1d(axes)
        for ax, (_, sampling_case) in zip(axes, sampling_cases.iterrows()):
            subset = summary[summary['sampling_condition'] == sampling_case['sampling_condition']]
            for idx, (_, recon_case) in enumerate(recon_cases.iterrows()):
                group = subset[subset['reconstruction_condition'] == recon_case['reconstruction_condition']]
                if group.empty:
                    continue
                group = group.sort_values('samp_perc', kind='stable')
                x = group['samp_perc'].to_numpy(dtype=float)
                y = group[metric].to_numpy(dtype=float)
                ci_halfwidth = group[band_column].fillna(0.0).to_numpy(dtype=float)
                color = reconstruction_color(recon_case['reconstruction_condition'], idx)
                label = recovery_math_label(recon_case['reconstruction_condition'], recon_case['reconstruction_label'])
                ax.plot(
                    x,
                    y,
                    label=label,
                    color=color,
                    marker=exp.SD15_RECON_MARKERS[idx % len(exp.SD15_RECON_MARKERS)],
                    markerfacecolor='white',
                    markeredgewidth=SWEEP_MARKER_EDGEWIDTH,
                    markersize=SWEEP_MARKERSIZE,
                    linewidth=SWEEP_LINEWIDTH,
                )
                if SHOW_METRIC_CONFIDENCE_INTERVALS and np.any(ci_halfwidth > 0.0):
                    ax.fill_between(x, y - ci_halfwidth, y + ci_halfwidth, color=color, alpha=0.16, linewidth=0)
            zero_group = zero_summary[zero_summary['sampling_condition'] == sampling_case['sampling_condition']]
            if not zero_group.empty:
                zero_group = zero_group.sort_values('samp_perc', kind='stable')
                x_zero = zero_group['samp_perc'].to_numpy(dtype=float)
                y_zero = zero_group[metric].to_numpy(dtype=float)
                zero_ci_halfwidth = zero_group[band_column].fillna(0.0).to_numpy(dtype=float)
                ax.plot(
                    x_zero,
                    y_zero,
                    label='Zero-Filled',
                    color='black',
                    linestyle='--',
                    marker='x',
                    markeredgewidth=SWEEP_MARKER_EDGEWIDTH,
                    markersize=SWEEP_MARKERSIZE,
                    linewidth=SWEEP_LINEWIDTH,
                )
                if SHOW_METRIC_CONFIDENCE_INTERVALS and np.any(zero_ci_halfwidth > 0.0):
                    ax.fill_between(x_zero, y_zero - zero_ci_halfwidth, y_zero + zero_ci_halfwidth, color='black', alpha=0.10, linewidth=0)
            ticks = sorted({float(value) for value in subset['samp_perc'].tolist()})
            ax.set_xscale('log')
            ax.set_xticks(ticks)
            ax.set_xticklabels(sampling_tick_labels(ticks), rotation=35, ha='right')
            ax.set_xlabel('')
            ax.set_title(sampling_mu_label(sampling_case['sampling_condition']))
            ax.grid(True, which='major', axis='both', alpha=0.28, linestyle='--')
        axes[0].set_ylabel(METRIC_LABELS.get(metric, title_case(metric)))
        fig.subplots_adjust(
            left=SWEEP_SINGLE_LEFT,
            right=SWEEP_SINGLE_RIGHT,
            bottom=SWEEP_SINGLE_BOTTOM,
            top=SWEEP_SINGLE_TOP,
            wspace=SWEEP_SINGLE_WSPACE,
        )
        fig.supxlabel(
            GLOBAL_SAMPLING_X_LABEL,
            fontsize=exp.SD15_PRESENTATION_RC.get('axes.labelsize', 30),
            y=0.045,
        )
        legend_handles = []
        legend_labels = []
        for legend_ax in axes:
            handles, labels = legend_ax.get_legend_handles_labels()
            for handle, label in zip(handles, labels):
                if label and label not in legend_labels:
                    legend_handles.append(handle)
                    legend_labels.append(label)
        if legend_handles:
            legend_handles, legend_labels = legend_zero_filled_last(legend_handles, legend_labels)
            fig.legend(
                legend_handles,
                legend_labels,
                loc='upper center',
                bbox_to_anchor=(0.5, SWEEP_SINGLE_LEGEND_Y),
                ncol=min(len(legend_labels), 5),
                frameon=False,
                fontsize=exp.SD15_PRESENTATION_RC.get('legend.fontsize', 22),
            )
        if output_path is not None:
            output_path = Path(output_path)
            output_path.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(output_path, dpi=exp.SD15_EXPORT_DPI, bbox_inches='tight')
        if show:
            plt.show()
        plt.close(fig)
        return output_path



def plot_combined_metric_curves(frame, metrics, output_path=None, show=True):
    if frame.empty:
        print('No rows available for the combined metric sweep.')
        return None

    summary = metric_summary(frame)
    sampling_cases = (
        summary[['sampling_condition', 'sampling_rank']]
        .drop_duplicates()
        .sort_values('sampling_rank', kind='stable')
    )
    recon_cases = (
        summary[['reconstruction_condition', 'reconstruction_label', 'recon_rank']]
        .drop_duplicates()
        .sort_values('recon_rank', kind='stable')
    )
    if sampling_cases.empty or recon_cases.empty:
        print('No cases available for the combined metric sweep.')
        return None

    n_rows = len(metrics)
    n_cols = len(sampling_cases)
    with plt.rc_context(exp.SD15_PRESENTATION_RC):
        fig, axes = plt.subplots(
            n_rows,
            n_cols,
            figsize=(SWEEP_FIGSIZE_PER_COL * n_cols, SWEEP_FIGSIZE_PER_ROW * n_rows),
            sharex='col',
            sharey='row',
            squeeze=False,
            constrained_layout=True,
        )
        for row_idx, metric in enumerate(metrics):
            band_column = f'{metric}_ci_halfwidth'
            zero_summary = zero_filled_metric_summary(frame, metric)
            for col_idx, (_, sampling_case) in enumerate(sampling_cases.iterrows()):
                ax = axes[row_idx, col_idx]
                subset = summary[summary['sampling_condition'] == sampling_case['sampling_condition']]
                for idx, (_, recon_case) in enumerate(recon_cases.iterrows()):
                    group = subset[subset['reconstruction_condition'] == recon_case['reconstruction_condition']]
                    if group.empty:
                        continue
                    group = group.sort_values('samp_perc', kind='stable')
                    x = group['samp_perc'].to_numpy(dtype=float)
                    y = group[metric].to_numpy(dtype=float)
                    ci_halfwidth = group[band_column].fillna(0.0).to_numpy(dtype=float)
                    color = reconstruction_color(recon_case['reconstruction_condition'], idx)
                    label = recovery_math_label(recon_case['reconstruction_condition'], recon_case['reconstruction_label'])
                    ax.plot(
                        x,
                        y,
                        label=label,
                        color=color,
                        marker=exp.SD15_RECON_MARKERS[idx % len(exp.SD15_RECON_MARKERS)],
                        markerfacecolor='white',
                        markeredgewidth=SWEEP_MARKER_EDGEWIDTH,
                        markersize=SWEEP_MARKERSIZE,
                        linewidth=SWEEP_LINEWIDTH,
                    )
                    if SHOW_METRIC_CONFIDENCE_INTERVALS and np.any(ci_halfwidth > 0.0):
                        ax.fill_between(x, y - ci_halfwidth, y + ci_halfwidth, color=color, alpha=0.16, linewidth=0)
                zero_group = zero_summary[zero_summary['sampling_condition'] == sampling_case['sampling_condition']]
                if not zero_group.empty:
                    zero_group = zero_group.sort_values('samp_perc', kind='stable')
                    x_zero = zero_group['samp_perc'].to_numpy(dtype=float)
                    y_zero = zero_group[metric].to_numpy(dtype=float)
                    zero_ci_halfwidth = zero_group[band_column].fillna(0.0).to_numpy(dtype=float)
                    ax.plot(
                        x_zero,
                        y_zero,
                        label='Zero-Filled',
                        color='black',
                        linestyle='--',
                        marker='x',
                        markeredgewidth=SWEEP_MARKER_EDGEWIDTH,
                        markersize=SWEEP_MARKERSIZE,
                        linewidth=SWEEP_LINEWIDTH,
                    )
                    if SHOW_METRIC_CONFIDENCE_INTERVALS and np.any(zero_ci_halfwidth > 0.0):
                        ax.fill_between(x_zero, y_zero - zero_ci_halfwidth, y_zero + zero_ci_halfwidth, color='black', alpha=0.10, linewidth=0)
                ticks = sorted({float(value) for value in subset['samp_perc'].tolist()})
                ax.set_xscale('log')
                ax.set_xticks(ticks)
                ax.set_xticklabels(sampling_tick_labels(ticks), rotation=35, ha='right')
                ax.set_xlabel('')
                if row_idx == 0:
                    ax.set_title(sampling_mu_label(sampling_case['sampling_condition']))
                if col_idx == 0:
                    ax.set_ylabel(METRIC_LABELS.get(metric, title_case(metric)))
                ax.grid(True, which='major', axis='both', alpha=0.28, linestyle='--')

        fig.supxlabel(GLOBAL_SAMPLING_X_LABEL, fontsize=exp.SD15_PRESENTATION_RC.get('axes.labelsize', 30))
        legend_handles = []
        legend_labels = []
        for legend_ax in axes.ravel():
            handles, labels = legend_ax.get_legend_handles_labels()
            for handle, label in zip(handles, labels):
                if label and label not in legend_labels:
                    legend_handles.append(handle)
                    legend_labels.append(label)
        if legend_handles:
            legend_handles, legend_labels = legend_zero_filled_last(legend_handles, legend_labels)
            fig.legend(
                legend_handles,
                legend_labels,
                loc='upper center',
                bbox_to_anchor=(0.5, SWEEP_LEGEND_Y),
                ncol=min(len(legend_labels), 5),
                frameon=False,
                fontsize=exp.SD15_PRESENTATION_RC.get('legend.fontsize', 22),
            )
        if output_path is not None:
            output_path = Path(output_path)
            output_path.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(output_path, dpi=exp.SD15_EXPORT_DPI, bbox_inches='tight')
        if show:
            plt.show()
        plt.close(fig)
        return output_path

if not ROWS.empty:
    for dc_method in ROWS['dc_method'].drop_duplicates().tolist():
        for sampling_method in ROWS['sampling_method'].drop_duplicates().tolist():
            subset = ROWS[(ROWS['dc_method'] == dc_method) & (ROWS['sampling_method'] == sampling_method)].copy()
            if subset.empty:
                continue
            sweep_metrics = ['psnr_db', 'ssim', 'pixel_mae']
            combined_metrics = ['psnr_db', 'ssim']
            for metric in sweep_metrics:
                plot_metric_curves(
                    subset,
                    metric,
                    output_path=OUTPUT_DIR / f'{dc_method}_{sampling_method}_{metric}_by_recovery_prompt.pdf',
                    show=True,
                )
            plot_combined_metric_curves(
                subset,
                combined_metrics,
                output_path=OUTPUT_DIR / f'{dc_method}_{sampling_method}_combined_metrics_by_recovery_prompt.pdf',
                show=True,
            )


## Recovery Grid

This cell builds the image grids used to inspect reconstruction quality directly. For each sampling prior, it selects one target item and one sampling ratio, then shows the ground truth, the zero-filled inverse FFT baseline, and the best available reconstruction for each recovery prompt.

The "best" reconstruction in each tile is selected from the loaded rows by PSNR first and SSIM second, so the grid is a compact visual counterpart to the metric curves. The cell saves one PDF per sampling prior in `OUTPUT_DIR` and displays the figures inline.

In [ ]:

from matplotlib import image as mpimg

IMAGE_REPEAT_ID = None
IMAGE_ITEM_ID = 0
IMAGE_DC_METHOD = DC_METHODS[0]
IMAGE_SAMPLING_METHOD = SAMPLING_METHODS[0]
IMAGE_SAMPLING_PERC = 0.00125
IMAGE_PANEL_WIDTH_IN = 3.0
IMAGE_PANEL_HEIGHT_IN = 4.2
IMAGE_SAMPLING_CONDITIONS = None
IMAGE_RECON_SELECTION_METRICS = ['psnr_db', 'ssim']


def sample_tag(value):
    return f'samp_{float(value):.5f}'.replace('.', 'p')


def run_artifact_dir(row):
    return (
        SD15_ROOT / 'results' / str(row['run_tag']) / str(row['sampling_method']) /
        f"item_{int(row['item_id']):03d}" / sample_tag(float(row['samp_perc'])) / f"rep_{int(row['repeat_id']):02d}"
    )


def recon_path_for(row):
    return run_artifact_dir(row) / f"recon_{row['sampling_method']}.png"


def zero_filled_path_for(row):
    return run_artifact_dir(row) / 'zero_filled_ifft.png'


def load_target_path(frame):
    run_tag = str(frame['run_tag'].iloc[0])
    dataset_ref_path = SD15_ROOT / 'results' / run_tag / 'dataset_ref.json'
    with dataset_ref_path.open('r', encoding='utf-8') as handle:
        dataset_ref = json.load(handle)
    item = next(item for item in dataset_ref['items'] if int(item['item_id']) == int(IMAGE_ITEM_ID))
    return Path(item['gt_png_path'])



def add_zero_filled_metric_label(ax, row):
    psnr = row.get('zero_filled_psnr_db', np.nan)
    ssim = row.get('zero_filled_ssim', np.nan)
    ppmae = row.get('zero_filled_pixel_mae', np.nan)
    if pd.isna(psnr) or pd.isna(ssim) or pd.isna(ppmae):
        return
    ax.text(
        0.02,
        0.96,
        f"PSNR {float(psnr):.2f} dB\nSSIM {float(ssim):.3f}\nPPMAE {float(ppmae):.4f}",
        transform=ax.transAxes,
        ha='left',
        va='top',
        fontsize=12,
        color='white',
        bbox={'boxstyle': 'square,pad=0.22', 'facecolor': (0, 0, 0, 0.46), 'edgecolor': 'none'},
    )

def show_image_or_placeholder(ax, path, *, cmap=None):
    path = Path(path)
    if path.is_file():
        ax.imshow(mpimg.imread(path), cmap=cmap)
    else:
        ax.text(0.5, 0.5, 'Missing', ha='center', va='center', transform=ax.transAxes, color='#6B7280')
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


def best_reconstruction_record(rows):
    sort_columns = [column for column in IMAGE_RECON_SELECTION_METRICS if column in rows.columns]
    tie_breakers = [column for column in ['item_id', 'repeat_id', 'run_tag'] if column in rows.columns]
    if sort_columns or tie_breakers:
        rows = rows.sort_values(
            sort_columns + tie_breakers,
            ascending=([False] * len(sort_columns)) + ([True] * len(tie_breakers)),
            na_position='last',
            kind='stable',
        )
    return rows.iloc[0]


def plot_recovery_grid(frame, *, sampling_condition, output_path=None, show=True):
    if frame.empty:
        print('No rows available for the recovery grid.')
        return None

    panel = frame[
        (frame['dc_method'] == IMAGE_DC_METHOD)
        & (frame['sampling_method'] == IMAGE_SAMPLING_METHOD)
        & (frame['sampling_condition'] == sampling_condition)
    ].copy()
    if IMAGE_ITEM_ID is not None:
        panel = panel[panel['item_id'] == IMAGE_ITEM_ID]
    if IMAGE_REPEAT_ID is not None:
        panel = panel[panel['repeat_id'] == IMAGE_REPEAT_ID]
    if panel.empty:
        print(f'No matching rows for sampling distribution {sampling_condition}.')
        return None

    panel = panel.sort_values(['samp_perc', 'recon_rank'], kind='stable')
    target_samp_perc = float(IMAGE_SAMPLING_PERC)
    keep_rate = np.isclose(panel['samp_perc'].astype(float).to_numpy(), target_samp_perc, rtol=0.0, atol=5e-8)
    panel = panel[keep_rate].copy()
    if panel.empty:
        print(f'No matching rows for sampling distribution {sampling_condition} at samp_perc={target_samp_perc:.5f}.')
        return None
    sampling_rates = [target_samp_perc]
    recon_cases = (
        panel[['reconstruction_condition', 'reconstruction_label', 'recon_rank']]
        .drop_duplicates()
        .sort_values('recon_rank', kind='stable')
    )
    target_path = load_target_path(panel)
    n_cols = 2 + len(recon_cases)

    with plt.rc_context(exp.SD15_PRESENTATION_RC):
        fig, axes = plt.subplots(
            len(sampling_rates),
            n_cols,
            figsize=(IMAGE_PANEL_WIDTH_IN * n_cols, IMAGE_PANEL_HEIGHT_IN * len(sampling_rates)),
            squeeze=False,
            constrained_layout=True,
        )
        fig.set_constrained_layout_pads(w_pad=0.02, h_pad=0.02, wspace=0.02, hspace=0.02)
        fig.suptitle(
            rf'Sampling Distribution: {sampling_mu_label(sampling_condition)}',
            fontsize=34,
        )
        for row_idx, samp_perc in enumerate(sampling_rates):
            rate_rows = panel[np.isclose(panel['samp_perc'].astype(float), float(samp_perc))]
            reference_row = best_reconstruction_record(rate_rows)
            show_image_or_placeholder(axes[row_idx, 0], target_path)
            show_image_or_placeholder(axes[row_idx, 1], zero_filled_path_for(reference_row))
            add_zero_filled_metric_label(axes[row_idx, 1], reference_row)
            axes[row_idx, 0].set_ylabel(f'{float(samp_perc):.5f}', fontsize=22, labelpad=4)
            if row_idx == 0:
                axes[row_idx, 0].set_title('Ground Truth', fontsize=22, pad=6)
                axes[row_idx, 1].set_title('Zero-Filled', fontsize=22, pad=6)
            for col_offset, (_, recon_case) in enumerate(recon_cases.iterrows(), start=2):
                match = rate_rows[rate_rows['reconstruction_condition'] == recon_case['reconstruction_condition']]
                ax = axes[row_idx, col_offset]
                if match.empty:
                    show_image_or_placeholder(ax, Path('__missing__'))
                    continue
                record = best_reconstruction_record(match)
                show_image_or_placeholder(ax, recon_path_for(record))
                if row_idx == 0:
                    ax.set_title(recovery_math_label(recon_case['reconstruction_condition'], recon_case['reconstruction_label']), fontsize=22, pad=6)
                ax.text(
                    0.02,
                    0.96,
                    f"PSNR {float(record['psnr_db']):.2f} dB\nSSIM {float(record['ssim']):.3f}\nPPMAE {float(record['pixel_mae']):.4f}",
                    transform=ax.transAxes,
                    ha='left',
                    va='top',
                    fontsize=12,
                    color='white',
                    bbox={'boxstyle': 'square,pad=0.22', 'facecolor': (0, 0, 0, 0.46), 'edgecolor': 'none'},
                )
        if output_path is not None:
            output_path = Path(output_path)
            output_path.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(output_path, dpi=exp.SD15_EXPORT_DPI, bbox_inches='tight')
        if show:
            plt.show()
        plt.close(fig)
        return output_path

if not ROWS.empty:
    image_sampling_cases = (
        ROWS[['sampling_condition', 'sampling_rank']]
        .drop_duplicates()
        .sort_values('sampling_rank', kind='stable')
    )
    if IMAGE_SAMPLING_CONDITIONS is not None:
        keep = {str(value) for value in IMAGE_SAMPLING_CONDITIONS}
        image_sampling_cases = image_sampling_cases[image_sampling_cases['sampling_condition'].isin(keep)]
    for _, sampling_case in image_sampling_cases.iterrows():
        sampling_condition = str(sampling_case['sampling_condition'])
        plot_recovery_grid(
            ROWS,
            sampling_condition=sampling_condition,
            output_path=OUTPUT_DIR / f'recovery_image_grid_{sampling_condition}.pdf',
            show=True,
        )
